# 学习一个 latent world

我们用 4 段 PixelWorld episode 接起 CNN Encoder、RSSM prior/posterior 与三个预测 head。CPU smoke 只验证 shape、梯度和 loss 下降。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

try:
    import torch
except ImportError as error:
    raise RuntimeError('请先安装 requirements-neural.txt') from error

from hwm.data import make_pixelworld_dataset
from hwm.neural import TinyWorldModel, batch_from_episodes, world_model_loss
torch.manual_seed(0)
print('PyTorch:', torch.__version__, 'device: cpu')

## 1. 从连续 episode 构造 batch

第 `t` 个动作对应 `observations[t] → observations[t+1]`。

In [ ]:
episodes = make_pixelworld_dataset(num_episodes=4, length=8, seed=0)
observations, actions, rewards, dones = batch_from_episodes(
    episodes, sequence_length=8
)
print('observations:', tuple(observations.shape), observations.dtype)
print('actions:     ', tuple(actions.shape), actions.dtype)
print('rewards:     ', tuple(rewards.shape))
assert observations.shape[:2] == (4, 9) and actions.shape == (4, 8)

## 2. 一次前向同时产生哪些结果

Posterior 看到了真实下一观察的 embedding；prior 只看历史状态和动作。

In [ ]:
model = TinyWorldModel()
loss, metrics, outputs = world_model_loss(
    model, observations, actions, rewards, dones
)
print('feature:', tuple(outputs['feature'].shape), '[B,T,deter+stoch]')
print('reconstruction:', tuple(outputs['reconstruction'].shape))
print('reward:', tuple(outputs['reward'].shape))
print('losses:', {k: round(float(v.detach()), 4) for k, v in metrics.items()})
assert outputs['feature'].shape == (4, 8, 80)

## 3. 小批数据上的过拟合检查

教学 smoke 故意重复同一小批数据。若 loss 连下降都做不到，先修代码和 shape，不急着收集更多数据。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for update in range(15):
    optimizer.zero_grad()
    loss, metrics, outputs = world_model_loss(
        model, observations, actions, rewards, dones
    )
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
    optimizer.step()
    losses.append(float(loss.detach()))
print('初始/最终 loss:', round(losses[0], 4), round(losses[-1], 4))
print('最后一次梯度 norm:', round(float(grad_norm.detach()), 4))
assert losses[-1] < losses[0]

Loss 下降只说明这批数据上的优化路径连通。它没有证明多步世界正确，也没有证明模型能帮助行动。

## 4. Prior 与 posterior 是否相同

In [ ]:
prior_mean = outputs['prior'].mean[0, -1]
posterior_mean = outputs['posterior'].mean[0, -1]
distance = torch.linalg.vector_norm(prior_mean - posterior_mean)
print('最后一步 prior/posterior mean 距离:', round(float(distance.detach()), 4))
print('KL:', round(float(metrics['kl']), 4))
assert torch.isfinite(distance)

Posterior 用真实图片修正状态，prior 学习在没有未来图片时靠动作预测。第二份 Notebook 会从 posterior 出发，只用 prior 想象。

## 小结

- [ ] Encoder 把像素变成 embedding。
- [ ] RSSM prior 与 posterior 的信息来源不同。
- [ ] Decoder、reward、continue 与 KL 约束不同信息。
- [ ] 小批过拟合是代码检查，不是研究结果。